# Tech Challenge — Fase 2
## Projeto 2: Otimização de Rotas Médicas com Algoritmos Genéticos e LLM

Este notebook foi preparado para:

- explicar a arquitetura da solução;
- demonstrar o TSP e o cenário hospitalar/VRP;
- executar o algoritmo genético e a abordagem heurística;
- comparar custo, distância, atraso, entregas não atendidas e tempo;
- executar pelo menos três configurações do algoritmo genético;
- visualizar a evolução do fitness e as rotas;
- demonstrar, opcionalmente, a integração com uma LLM.

> Execute este notebook a partir da **raiz do repositório**, no mesmo ambiente virtual utilizado pela aplicação.


## 1. Relação com os requisitos do Projeto 2

| Requisito | Implementação demonstrada |
|---|---|
| Representação genética de rotas | Cromossomo representado por uma permutação das entregas/cidades |
| Seleção, cruzamento e mutação | Seleção do algoritmo, crossover OX/OX1 e mutação por troca |
| Função fitness | Distância, prioridade, atraso, entregas não atendidas e balanceamento |
| Prioridades | Entregas regulares, altas e críticas |
| Capacidade | Limite de carga por veículo |
| Autonomia | Distância máxima por veículo |
| Múltiplos veículos | Extensão do TSP para VRP hospitalar |
| Visualização | Aplicação Pygame e gráficos neste notebook |
| Comparação | Algoritmo genético versus heurística |
| Três experimentos | Três configurações de população, mutação e elitismo |
| Integração com LLM | Geração de análise textual a partir do resultado consolidado |


In [ ]:
from pathlib import Path
import sys
import time
import json

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "execution").exists() and (PROJECT_ROOT.parent / "execution").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raiz do projeto:", PROJECT_ROOT)
print("Python:", sys.executable)


In [ ]:
# Caso necessário:
# python -m pip install pandas matplotlib jupyter

import pandas as pd
import matplotlib.pyplot as plt

from execution import (
    ExecutionConfig,
    ExperimentRunner,
    ProcessingState,
    ProblemType,
    AlgorithmMode,
    ExperimentRequest,
)
from scenarios import list_scenarios, load_scenario
from benchmark_att48 import att_48_cities_locations

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("Importações concluídas.")


## 2. Arquitetura resumida

```text
Cenário JSON / ATT48
        ↓
ExperimentRequest + ExecutionConfig
        ↓
ExperimentRunner
        ↓
Algoritmo Genético ou Heurística
        ↓
Solução consolidada
        ↓
Interface / Exportação / Notebook / LLM
```

### Componentes principais

- `execution`: configuração, modelos, consolidação e controle das execuções;
- `optimizers`: algoritmo genético e abordagem heurística;
- `routing`: avaliação das rotas hospitalares e função objetivo;
- `scenarios`: leitura e validação dos cenários JSON;
- `ui`: interface Pygame;
- `services`: integração opcional com a LLM;
- `results`: exportações JSON e CSV.


In [ ]:
def run_request(request, step_size=50, timeout_seconds=300):
    # Executa o ExperimentRunner até conclusão, erro ou timeout.
    runner = ExperimentRunner()
    runner.start(request)
    started = time.perf_counter()

    while runner.state in (ProcessingState.RUNNING, ProcessingState.PAUSED):
        if runner.state is ProcessingState.PAUSED:
            runner.resume()

        runner.step(step_size)

        if time.perf_counter() - started > timeout_seconds:
            runner.cancel()
            raise TimeoutError(
                f"Execução excedeu {timeout_seconds} segundos e foi cancelada."
            )

    if runner.state is ProcessingState.ERROR:
        raise RuntimeError(runner.message)

    return runner


def hospital_solution_summary(solution):
    delivered = sum(len(route.stops) for route in solution.routes)
    vehicles_used = sum(bool(route.stops) for route in solution.routes)

    return {
        "custo_objetivo": solution.objective_cost,
        "distancia_km": solution.total_distance_km,
        "duracao_min": solution.total_duration_minutes,
        "atraso_min": solution.total_delay_minutes,
        "entregas_atendidas": delivered,
        "nao_atendidas": len(solution.unassigned_deliveries),
        "veiculos_utilizados": vehicles_used,
        "viavel": solution.is_feasible,
    }


def summarize_runner(runner, label=None):
    snapshot = runner.get_snapshot()
    result = runner.get_result()
    comparison = runner.get_comparison()
    rows = []

    if snapshot.problem_type is ProblemType.TSP:
        sources = (
            [("Genético", comparison.genetic), ("Vizinho Mais Próximo", comparison.nearest)]
            if comparison
            else [(label or result.algorithm, result)]
        )

        for algorithm, experiment in sources:
            rows.append({
                "algoritmo": algorithm,
                "melhor_distancia": experiment.best_run.best_fitness,
                "distancia_media": experiment.average_fitness,
                "pior_distancia": experiment.worst_fitness,
                "desvio_padrao": experiment.standard_deviation,
                "tempo_medio_s": experiment.average_elapsed_seconds,
                "execucoes": len(experiment.runs),
            })
        return pd.DataFrame(rows)

    sources = (
        [("Genético", comparison.genetic), ("Heurística", comparison.heuristic)]
        if comparison
        else [(label or result.algorithm, result)]
    )

    for algorithm, experiment in sources:
        rows.append({
            "algoritmo": algorithm,
            **hospital_solution_summary(experiment.best_run.solution),
            "custo_medio": experiment.average_objective_cost,
            "pior_custo": experiment.worst_objective_cost,
            "desvio_padrao": experiment.standard_deviation,
            "tempo_medio_s": experiment.average_elapsed_seconds,
            "execucoes": len(experiment.runs),
            "melhor_semente": experiment.best_run.seed,
        })

    return pd.DataFrame(rows)


## 3. Cenários hospitalares disponíveis

Os cenários podem representar situações viáveis, prioridades, prazos apertados, frota heterogênea, capacidade insuficiente, autonomia insuficiente, entregas impossíveis e alta demanda.

Isso permite demonstrar que a função objetivo não considera apenas distância.


In [ ]:
scenario_paths = list_scenarios()

scenario_catalog = []
for path in scenario_paths:
    scenario = load_scenario(path)
    scenario_catalog.append({
        "arquivo": path.name,
        "id": scenario.id,
        "nome": scenario.name,
        "entregas": len(scenario.deliveries),
        "veiculos": len(scenario.vehicles),
    })

df_scenarios = pd.DataFrame(scenario_catalog)
df_scenarios


## 4. Demonstração do TSP com ATT48

O TSP clássico procura uma sequência que visite todas as cidades e minimize a distância total.

No projeto:

- o benchmark ATT48 fornece 48 coordenadas;
- o algoritmo genético trabalha com permutações;
- o crossover preserva uma subsequência e completa o restante sem repetir cidades;
- a mutação troca posições;
- o resultado é comparado com o Vizinho Mais Próximo.


In [ ]:
TSP_CONFIG = ExecutionConfig(
    algorithm="Comparar ambos",
    population_size=100,
    generations=300,
    executions=3,
    mutation_probability=0.10,
    elite_count=2,
    random_seed=42,
)

tsp_request = ExperimentRequest(
    problem_type=ProblemType.TSP,
    algorithm_mode=AlgorithmMode.COMPARE,
    execution_config=TSP_CONFIG,
    tsp_cities=tuple(att_48_cities_locations),
)

tsp_runner = run_request(tsp_request, step_size=100)
df_tsp = summarize_runner(tsp_runner)
df_tsp


In [ ]:
comparison = tsp_runner.get_comparison()

if comparison:
    print(f"Melhor distância do AG: {comparison.genetic.best_run.best_fitness:,.2f}")
    print(f"Distância do Vizinho Mais Próximo: {comparison.nearest.best_run.best_fitness:,.2f}")
    print(f"Melhoria percentual do AG: {comparison.improvement_percentage:,.2f}%")


In [ ]:
def plot_tsp_route(route, title):
    points = list(route)
    if points and points[0] != points[-1]:
        points = points + [points[0]]

    x = [p[0] for p in points]
    y = [p[1] for p in points]

    plt.figure(figsize=(10, 7))
    plt.plot(x, y, marker="o", linewidth=1)
    plt.scatter(x[:1], y[:1], s=100, label="Início")
    plt.title(title)
    plt.xlabel("Coordenada X")
    plt.ylabel("Coordenada Y")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


if comparison:
    plot_tsp_route(
        comparison.genetic.best_run.best_route,
        "TSP ATT48 — Melhor rota do Algoritmo Genético",
    )
    plot_tsp_route(
        comparison.nearest.best_run.best_route,
        "TSP ATT48 — Rota do Vizinho Mais Próximo",
    )


In [ ]:
if comparison:
    history = comparison.genetic.best_run.fitness_history

    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(history) + 1), history)
    plt.title("Evolução do melhor fitness — TSP")
    plt.xlabel("Geração")
    plt.ylabel("Distância")
    plt.grid(True, alpha=0.3)
    plt.show()


## 5. Demonstração hospitalar/VRP

No cenário hospitalar, o cromossomo representa a ordem de atendimento, e um decodificador distribui as entregas entre os veículos.

```text
custo objetivo =
    distância
  + custo de prioridade
  + penalidade de atraso
  + penalidade de capacidade
  + penalidade de autonomia
  + penalidade de entrega não atendida
  + balanceamento das rotas
```


In [ ]:
preferred_terms = ("alta_demanda", "prioridades", "prazos_apertados")

selected_path = None
for term in preferred_terms:
    selected_path = next((p for p in scenario_paths if term in p.stem.lower()), None)
    if selected_path:
        break

if selected_path is None:
    selected_path = scenario_paths[0]

hospital_scenario = load_scenario(selected_path)

print("Cenário selecionado:", hospital_scenario.name)
print("Arquivo:", selected_path)
print("Entregas:", len(hospital_scenario.deliveries))
print("Veículos:", len(hospital_scenario.vehicles))


In [ ]:
HOSPITAL_CONFIG = ExecutionConfig(
    algorithm="Comparar ambos",
    population_size=120,
    generations=400,
    executions=3,
    mutation_probability=0.15,
    elite_count=3,
    random_seed=100,
)

hospital_request = ExperimentRequest(
    problem_type=ProblemType.HOSPITAL,
    algorithm_mode=AlgorithmMode.COMPARE,
    execution_config=HOSPITAL_CONFIG,
    hospital_scenario=hospital_scenario,
)

hospital_runner = run_request(hospital_request, step_size=100)
df_hospital_comparison = summarize_runner(hospital_runner)
df_hospital_comparison


In [ ]:
comparison = hospital_runner.get_comparison()

if comparison:
    print(f"Melhoria do AG no custo objetivo: {comparison.improvement_percentage:,.2f}%")
    print(f"Diferença absoluta de custo: {comparison.objective_difference:,.2f}")
    print(f"Diferença de distância: {comparison.distance_difference_km:,.2f} km")
    print(f"Diferença de atraso: {comparison.delay_difference_minutes:,.2f} min")
    print(f"Diferença de entregas não atendidas: {comparison.unassigned_difference}")


In [ ]:
def route_rows(solution):
    rows = []
    for route in solution.routes:
        rows.append({
            "veiculo": route.vehicle.name,
            "paradas": len(route.stops),
            "carga_kg": route.total_load_kg,
            "capacidade_kg": route.vehicle.capacity_kg,
            "distancia_km": route.total_distance_km,
            "autonomia_km": route.vehicle.autonomy_km,
            "duracao_min": route.total_duration_minutes,
            "atraso_total_min": sum(stop.delay_minutes for stop in route.stops),
        })
    return pd.DataFrame(rows)


if comparison:
    print("Rotas do Algoritmo Genético")
    display(route_rows(comparison.genetic.best_run.solution))

    print("Rotas da Heurística")
    display(route_rows(comparison.heuristic.best_run.solution))


In [ ]:
def plot_hospital_solution(scenario, solution, title):
    plt.figure(figsize=(11, 8))

    depot_x, depot_y = scenario.depot.x, scenario.depot.y
    plt.scatter([depot_x], [depot_y], marker="s", s=160, label="Depósito")

    for route in solution.routes:
        if not route.stops:
            continue

        xs = [depot_x] + [stop.delivery.location.x for stop in route.stops] + [depot_x]
        ys = [depot_y] + [stop.delivery.location.y for stop in route.stops] + [depot_y]

        plt.plot(xs, ys, marker="o", linewidth=1.6, label=route.vehicle.name)

        for stop in route.stops:
            plt.annotate(
                stop.delivery.id,
                (stop.delivery.location.x, stop.delivery.location.y),
                fontsize=8,
            )

    plt.title(title)
    plt.xlabel("Coordenada X")
    plt.ylabel("Coordenada Y")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


if comparison:
    plot_hospital_solution(
        hospital_scenario,
        comparison.genetic.best_run.solution,
        "VRP Hospitalar — Melhor solução do Algoritmo Genético",
    )
    plot_hospital_solution(
        hospital_scenario,
        comparison.heuristic.best_run.solution,
        "VRP Hospitalar — Solução Heurística",
    )


In [ ]:
if comparison:
    history = comparison.genetic.best_run.history

    history_df = pd.DataFrame([
        {
            "geracao": metric.generation,
            "custo_objetivo": metric.objective_cost,
            "distancia_km": metric.total_distance_km,
            "atraso_min": metric.total_delay_minutes,
            "nao_atendidas": metric.unassigned_count,
            "veiculos_utilizados": metric.vehicles_used,
        }
        for metric in history
    ])

    display(history_df.tail())

    plt.figure(figsize=(10, 5))
    plt.plot(history_df["geracao"], history_df["custo_objetivo"])
    plt.title("Evolução do custo objetivo — VRP hospitalar")
    plt.xlabel("Geração")
    plt.ylabel("Custo objetivo")
    plt.grid(True, alpha=0.3)
    plt.show()


## 6. Três experimentos com configurações diferentes do AG

A tabela compara:

1. população menor e mutação conservadora;
2. configuração intermediária;
3. população maior, mais elitismo e mutação mais alta.

A semente base é controlada para facilitar a repetibilidade.


In [ ]:
EXPERIMENT_CONFIGS = [
    {
        "experimento": "AG-1 Econômico",
        "population_size": 60,
        "generations": 250,
        "executions": 3,
        "mutation_probability": 0.08,
        "elite_count": 1,
        "random_seed": 1000,
    },
    {
        "experimento": "AG-2 Equilibrado",
        "population_size": 120,
        "generations": 400,
        "executions": 3,
        "mutation_probability": 0.15,
        "elite_count": 3,
        "random_seed": 2000,
    },
    {
        "experimento": "AG-3 Exploração",
        "population_size": 200,
        "generations": 600,
        "executions": 3,
        "mutation_probability": 0.25,
        "elite_count": 5,
        "random_seed": 3000,
    },
]

# True: execução rápida durante a apresentação.
# False: usa integralmente as configurações acima.
MODO_RAPIDO = True


In [ ]:
experiment_rows = []
experiment_runners = {}

for item in EXPERIMENT_CONFIGS:
    cfg_data = dict(item)

    if MODO_RAPIDO:
        cfg_data["population_size"] = min(cfg_data["population_size"], 80)
        cfg_data["generations"] = min(cfg_data["generations"], 180)
        cfg_data["executions"] = 3
        cfg_data["elite_count"] = min(
            cfg_data["elite_count"],
            cfg_data["population_size"] - 1,
        )

    label = cfg_data.pop("experimento")

    config = ExecutionConfig(
        algorithm="Algoritmo Genético",
        **cfg_data,
    )

    request = ExperimentRequest(
        problem_type=ProblemType.HOSPITAL,
        algorithm_mode=AlgorithmMode.GENETIC,
        execution_config=config,
        hospital_scenario=hospital_scenario,
    )

    print("Executando:", label)
    runner = run_request(request, step_size=100)
    experiment_runners[label] = runner

    result = runner.get_result()
    best = result.best_run.solution

    experiment_rows.append({
        "experimento": label,
        "populacao": config.population_size,
        "geracoes": config.generations,
        "execucoes": config.executions,
        "mutacao": config.mutation_probability,
        "elitismo": config.elite_count,
        "melhor_custo": best.objective_cost,
        "custo_medio": result.average_objective_cost,
        "pior_custo": result.worst_objective_cost,
        "desvio_padrao": result.standard_deviation,
        "distancia_km": best.total_distance_km,
        "atraso_min": best.total_delay_minutes,
        "nao_atendidas": len(best.unassigned_deliveries),
        "tempo_medio_s": result.average_elapsed_seconds,
        "melhor_semente": result.best_run.seed,
    })

df_experiments = pd.DataFrame(experiment_rows).sort_values("melhor_custo")
df_experiments


In [ ]:
metrics_to_plot = [
    ("melhor_custo", "Melhor custo objetivo"),
    ("distancia_km", "Distância da melhor solução (km)"),
    ("atraso_min", "Atraso da melhor solução (min)"),
    ("tempo_medio_s", "Tempo médio por execução (s)"),
]

for column, title in metrics_to_plot:
    plt.figure(figsize=(9, 4))
    plt.bar(df_experiments["experimento"], df_experiments[column])
    plt.title(title)
    plt.ylabel(column)
    plt.xticks(rotation=15)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


### Interpretação esperada

- População maior aumenta a diversidade, mas eleva o custo computacional.
- Mais gerações dão mais oportunidade de convergência.
- Mutação muito baixa pode causar convergência prematura.
- Mutação muito alta pode destruir soluções promissoras.
- Elitismo preserva os melhores indivíduos, mas em excesso reduz diversidade.
- Uma única execução não é suficiente; média e desvio-padrão ajudam a medir estabilidade.


## 7. Comparação de todos os cenários

Esta seção é opcional. Para uma apresentação curta, deixe `EXECUTAR_TODOS_OS_CENARIOS = False`.


In [ ]:
EXECUTAR_TODOS_OS_CENARIOS = False
all_scenarios_rows = []

if EXECUTAR_TODOS_OS_CENARIOS:
    for scenario_path in scenario_paths:
        scenario = load_scenario(scenario_path)

        config = ExecutionConfig(
            algorithm="Comparar ambos",
            population_size=100,
            generations=300,
            executions=3,
            mutation_probability=0.15,
            elite_count=3,
            random_seed=5000,
        )

        request = ExperimentRequest(
            problem_type=ProblemType.HOSPITAL,
            algorithm_mode=AlgorithmMode.COMPARE,
            execution_config=config,
            hospital_scenario=scenario,
        )

        print("Executando cenário:", scenario.name)
        runner = run_request(request, step_size=100)
        scenario_comparison = runner.get_comparison()

        for algorithm, experiment in [
            ("Genético", scenario_comparison.genetic),
            ("Heurística", scenario_comparison.heuristic),
        ]:
            all_scenarios_rows.append({
                "cenario": scenario.name,
                "algoritmo": algorithm,
                **hospital_solution_summary(experiment.best_run.solution),
                "tempo_medio_s": experiment.average_elapsed_seconds,
            })

    df_all_scenarios = pd.DataFrame(all_scenarios_rows)
    display(df_all_scenarios)
else:
    print("Seção não executada. Altere EXECUTAR_TODOS_OS_CENARIOS para True.")


## 8. Integração com LLM

A LLM não calcula a rota:

```text
Algoritmos de otimização → calculam as rotas e métricas
LLM                       → interpreta, explica e recomenda
```

A chave deve ficar somente no arquivo `.env`, que não deve ser versionado.


In [ ]:
EXECUTAR_LLM = False

if EXECUTAR_LLM:
    from services.llm_analyzer import LlmAnalyzer, build_analysis_payload

    payload = build_analysis_payload(
        hospital_runner,
        hospital_scenario,
    )

    print("Resumo enviado para a LLM:")
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:5000])

    analyzer = LlmAnalyzer()
    analysis = analyzer.analyze(payload)

    print("\nANÁLISE GERADA PELA LLM\n")
    print(analysis)
else:
    print("Integração não executada. Altere EXECUTAR_LLM para True.")


## 9. Conclusões para apresentação

1. O código base do TSP foi evoluído para um problema hospitalar com múltiplos veículos.
2. A representação genética continua válida porque cada indivíduo é uma ordem de atendimento.
3. O decodificador transforma essa ordem em rotas por veículo.
4. A função objetivo considera distância e restrições hospitalares.
5. O algoritmo genético é comparado com uma heurística determinística.
6. Três configurações mostram o impacto dos hiperparâmetros.
7. Os gráficos permitem explicar convergência, distância, atraso e custo.
8. A LLM transforma as métricas em uma análise textual compreensível.

### Mensagem principal

O melhor resultado não é necessariamente a menor distância isolada. No contexto hospitalar, o sistema equilibra urgência, atraso, capacidade, autonomia, veículos, entregas não atendidas e distância.


## 10. Roteiro sugerido usando este notebook

1. Contexto e objetivo.
2. Arquitetura da solução.
3. TSP ATT48: AG versus Vizinho Mais Próximo.
4. VRP hospitalar: prioridades, capacidade, autonomia e múltiplos veículos.
5. Tabela comparativa.
6. Evolução do AG.
7. Três experimentos.
8. Interface Pygame.
9. Integração com LLM.
10. Conclusão.
